# 太阳黑子数时间序列分析与预测

本 notebook 分析1700年至今的太阳黑子数时间序列，通过：
1. **频谱分析** - 验证约11年的太阳活动周期
2. **ARIMA模型** - 预测下一太阳活动周峰值（改进版：BIC+交叉验证防止过拟合）

## 安装依赖
```bash
pip install numpy pandas matplotlib scipy statsmodels
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

## 1. 数据加载

数据来源：SILSO (Sunspot Index and Long-term Solar Observations)

In [ ]:
def load_sunspot_data():
    """加载太阳黑子数据"""
    try:
        url = "https://www.sidc.be/SILSO/DATA/SN_y_tot_V2.0.txt"
        data = pd.read_csv(url, sep='\s+', header=None, 
                          names=['Year', 'SunspotNumber', 'Std', 'N'], 
                          usecols=['Year', 'SunspotNumber'])
        data['Year'] = data['Year'].astype(int)
        data = data[data['Year'] >= 1700]
        print("成功加载在线数据")
        return data
    except:
        print("无法获取在线数据，使用模拟数据演示...")
        return generate_simulated_data()

def generate_simulated_data():
    """生成模拟的太阳黑子数据（基于11年周期）"""
    years = np.arange(1700, 2025)
    np.random.seed(42)
    
    cycle = 11.1
    phase = 2.0
    amplitude = 80
    trend = 0.05 * (years - 1700)
    noise = np.random.normal(0, 15, len(years))
    
    sunspots = amplitude * (0.5 + 0.5 * np.sin(2 * np.pi * (years - phase) / cycle)) + trend + noise
    sunspots = np.maximum(sunspots, 0)
    
    return pd.DataFrame({'Year': years, 'SunspotNumber': sunspots})

data = load_sunspot_data()
print(f"\n数据范围: {data['Year'].min()} - {data['Year'].max()} 年")
print(f"数据点数: {len(data)}")
print(f"平均太阳黑子数: {data['SunspotNumber'].mean():.2f}")
print(f"最大太阳黑子数: {data['SunspotNumber'].max():.2f} ({data.loc[data['SunspotNumber'].idxmax(), 'Year']}年)")

## 2. 历史太阳活动周分析

In [ ]:
def cycle_analysis(data):
    """分析历史太阳活动周特征"""
    years = data['Year'].values
    sunspots = data['SunspotNumber'].values
    
    peaks = []
    for i in range(5, len(sunspots) - 5):
        if sunspots[i] == max(sunspots[i-5:i+6]) and sunspots[i] > 50:
            peaks.append((years[i], sunspots[i]))
    
    peaks = np.array(peaks)
    if len(peaks) > 1:
        intervals = np.diff(peaks[:, 0])
        avg_interval = np.mean(intervals)
        avg_amplitude = np.mean(peaks[:, 1])
        
        print(f"检测到 {len(peaks)} 个历史峰值")
        print(f"平均周期间隔: {avg_interval:.2f} 年")
        print(f"平均峰值强度: {avg_amplitude:.1f}")
        print(f"\n最近5个太阳活动周:")
        for i in range(max(0, len(peaks)-5), len(peaks)):
            print(f"  峰值年份: {int(peaks[i, 0])}, 峰值: {peaks[i, 1]:.1f}")
    
    return peaks

peaks = cycle_analysis(data)

## 3. 频谱分析

使用Welch方法进行功率谱密度分析，检测主导周期。

In [ ]:
def spectral_analysis(data):
    """频谱分析检测周期"""
    years = data['Year'].values
    sunspots = data['SunspotNumber'].values
    n = len(sunspots)
    
    freqs, psd = signal.welch(sunspots, fs=1.0, nperseg=min(256, n))
    
    periods = 1 / freqs[freqs > 0]
    psd_pos = psd[freqs > 0]
    
    peak_idx = np.argmax(psd_pos)
    dominant_period = periods[peak_idx]
    
    print(f"检测到的主导周期: {dominant_period:.2f} 年")
    print(f"理论太阳活动周期: 约 11 年")
    
    fig, axes = plt.subplots(2, 1, figsize=(12, 10))
    
    axes[0].plot(years, sunspots, 'b-', linewidth=1)
    axes[0].set_xlabel('年份')
    axes[0].set_ylabel('太阳黑子数')
    axes[0].set_title('太阳黑子数年平均值 (1700-至今)')
    axes[0].grid(True, alpha=0.3)
    
    axes[1].semilogx(periods, psd_pos, 'r-', linewidth=1.5)
    axes[1].axvline(dominant_period, color='g', linestyle='--', 
                    label=f'主导周期: {dominant_period:.1f}年')
    axes[1].axvline(11, color='orange', linestyle=':', label='理论周期: 11年')
    axes[1].set_xlabel('周期 (年)')
    axes[1].set_ylabel('功率谱密度')
    axes[1].set_title('太阳黑子数功率谱分析')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    axes[1].set_xlim(2, 50)
    
    plt.tight_layout()
    plt.savefig('spectral_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    return dominant_period

dominant_period = spectral_analysis(data)

## 4. ARIMA模型预测（改进版：BIC+交叉验证防止过拟合）

### 4.1 确定差分阶数d (ADF检验)

In [ ]:
def check_stationarity(series, diff_level=0):
    """检查时间序列平稳性"""
    result = adfuller(series)
    if diff_level == 0:
        print("ADF平稳性检验 (原始序列):")
    else:
        print(f"ADF平稳性检验 ({diff_level}阶差分后):")
    print(f"ADF统计量: {result[0]:.4f}")
    print(f"p值: {result[1]:.4f}")
    print(f"临界值:")
    for key, value in result[4].items():
        print(f"  {key}: {value:.4f}")
    is_stationary = result[1] < 0.05
    print(f"结论: {'平稳 ✓' if is_stationary else '非平稳 ✗'}")
    return is_stationary

def determine_d(sunspots, max_d=2):
    """确定差分阶数d"""
    print("-"*60)
    print("确定差分阶数d (ADF检验)")
    print("-"*60)
    
    current_series = sunspots.copy()
    for d in range(max_d + 1):
        is_stationary = check_stationarity(current_series, diff_level=d)
        if is_stationary:
            print(f"\n选择差分阶数 d = {d}")
            return d
        if d < max_d:
            current_series = np.diff(current_series)
    
    print(f"\n达到最大差分阶数，选择 d = {max_d}")
    return max_d

sunspots = data['SunspotNumber'].values
d = determine_d(sunspots, max_d=2)

### 4.2 ARIMA定阶：BIC准则 + 交叉验证（防止过拟合）

In [ ]:
def arima_cross_validation(sunspots, order, n_splits=5, test_size=20):
    """时间序列交叉验证，返回平均RMSE"""
    n = len(sunspots)
    rmse_list = []
    
    for i in range(n_splits):
        split_idx = n - test_size * (n_splits - i)
        if split_idx < 10:
            continue
            
        train = sunspots[:split_idx]
        test = sunspots[split_idx:split_idx + test_size]
        
        if len(test) == 0:
            continue
            
        try:
            model = ARIMA(train, order=order)
            results = model.fit()
            forecast = results.get_forecast(steps=len(test)).predicted_mean
            rmse = np.sqrt(np.mean((forecast - test) ** 2))
            rmse_list.append(rmse)
        except:
            continue
    
    if len(rmse_list) == 0:
        return np.inf
    
    return np.mean(rmse_list)

def select_arima_order(sunspots, d, max_p=4, max_q=4):
    """使用BIC准则和交叉验证选择ARIMA阶数，防止过拟合"""
    print("\n" + "-"*60)
    print("ARIMA定阶: BIC准则 + 交叉验证 (防止过拟合)")
    print("-"*60)
    
    bic_results = []
    cv_results = []
    
    print("\n1. BIC准则筛选候选模型:")
    print(f"{'(p,d,q)':<12} {'BIC':<12} {'收敛':<8}")
    print("-" * 35)
    
    for p in range(0, max_p + 1):
        for q in range(0, max_q + 1):
            try:
                model = ARIMA(sunspots, order=(p, d, q))
                results = model.fit()
                if results.mle_retvals['converged']:
                    bic_results.append({
                        'order': (p, d, q),
                        'bic': results.bic,
                        'converged': True
                    })
                    print(f"({p},{d},{q}):    {results.bic:<10.2f} ✓")
                else:
                    bic_results.append({
                        'order': (p, d, q),
                        'bic': np.inf,
                        'converged': False
                    })
                    print(f"({p},{d},{q}):    {'-':<10} ✗ (未收敛)")
            except:
                bic_results.append({
                    'order': (p, d, q),
                    'bic': np.inf,
                    'converged': False
                })
                print(f"({p},{d},{q}):    {'-':<10} ✗ (失败)")
    
    bic_results.sort(key=lambda x: x['bic'])
    top_candidates = [r for r in bic_results if r['converged']][:5]
    
    print(f"\n2. BIC排名前5的模型:")
    for i, r in enumerate(top_candidates):
        print(f"  {i+1}. {r['order']}, BIC = {r['bic']:.2f}")
    
    print("\n3. 交叉验证验证预测能力 (滚动窗口):")
    print(f"{'(p,d,q)':<12} {'CV-RMSE':<12}")
    print("-" * 25)
    
    for r in top_candidates:
        cv_rmse = arima_cross_validation(sunspots, r['order'])
        cv_results.append({
            'order': r['order'],
            'bic': r['bic'],
            'cv_rmse': cv_rmse
        })
        print(f"{r['order']}:    {cv_rmse:.2f}")
    
    cv_results.sort(key=lambda x: x['cv_rmse'])
    best_order = cv_results[0]['order']
    
    print(f"\n4. 最优模型选择:")
    print(f"   BIC最优: {bic_results[0]['order']}, BIC = {bic_results[0]['bic']:.2f}")
    print(f"   CV最优:  {cv_results[0]['order']}, CV-RMSE = {cv_results[0]['cv_rmse']:.2f}")
    print(f"   最终选择: {best_order} (基于交叉验证)")
    
    return best_order, bic_results[0]['bic'], cv_results[0]['cv_rmse']

best_order, best_bic, best_cv_rmse = select_arima_order(sunspots, d, max_p=4, max_q=4)

### 4.3 拟合最终模型 + 残差诊断

In [ ]:
def residual_diagnostics(results, order):
    """残差诊断，验证模型拟合质量"""
    print("\n" + "-"*60)
    print("模型残差诊断")
    print("-" * 60)
    
    residuals = results.resid
    
    lb_test = acorr_ljungbox(residuals, lags=[10], return_df=True)
    p_value = lb_test['lb_pvalue'].values[0]
    
    print(f"Ljung-Box检验 (滞后10期):")
    print(f"  统计量: {lb_test['lb_stat'].values[0]:.4f}")
    print(f"  p值: {p_value:.4f}")
    
    if p_value > 0.05:
        print("  结论: 残差无显著自相关 ✓ (模型充分)")
    else:
        print("  结论: 残差存在自相关 ✗ (模型可能不充分)")
    
    print(f"\n残差统计:")
    print(f"  均值: {np.mean(residuals):.4f}")
    print(f"  标准差: {np.std(residuals):.4f}")

model = ARIMA(sunspots, order=best_order)
results = model.fit()

print("拟合最终模型")
print("-" * 60)
print(f"模型: ARIMA{best_order}")
print(f"BIC: {results.bic:.2f}")
print(f"AIC: {results.aic:.2f}")
print(f"对数似然: {results.llf:.2f}")

residual_diagnostics(results, best_order)

### 4.4 预测下一太阳活动周峰值

In [ ]:
forecast_years = 25
years = data['Year'].values
last_year = years[-1]

forecast = results.get_forecast(steps=forecast_years)
forecast_mean = forecast.predicted_mean
forecast_ci = forecast.conf_int()

future_years = np.arange(last_year + 1, last_year + forecast_years + 1)

peak_idx = np.argmax(forecast_mean)
peak_year = future_years[peak_idx]
peak_value = forecast_mean[peak_idx]

print("-"*60)
print("预测结果")
print("-" * 60)
print(f"下一太阳活动周峰值年份: {peak_year}")
print(f"预测峰值太阳黑子数: {peak_value:.1f}")
print(f"95%置信区间: [{forecast_ci[peak_idx, 0]:.1f}, {forecast_ci[peak_idx, 1]:.1f}]")

fig, ax = plt.subplots(figsize=(14, 7))

ax.plot(years, sunspots, 'b-', label='历史数据', linewidth=1.5, alpha=0.7)
ax.plot(future_years, forecast_mean, 'r-', label='预测值', linewidth=2)
ax.fill_between(future_years, 
                forecast_ci[:, 0], 
                forecast_ci[:, 1], 
                color='red', alpha=0.2, label='95%置信区间')

ax.scatter(peak_year, peak_value, color='gold', s=200, zorder=5, edgecolor='black', linewidth=2)
ax.annotate(f'预测峰值\n{peak_year}年\n{peak_value:.0f}', 
            xy=(peak_year, peak_value), 
            xytext=(peak_year + 3, peak_value + 20),
            fontsize=12, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.5', facecolor='yellow', alpha=0.8),
            arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0'))

model_text = f'ARIMA{best_order}\nBIC={best_bic:.0f}\nCV-RMSE={best_cv_rmse:.1f}'
ax.text(0.02, 0.98, model_text, transform=ax.transAxes, 
        fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

ax.set_xlabel('年份', fontsize=12)
ax.set_ylabel('太阳黑子数', fontsize=12)
ax.set_title('太阳黑子数时间序列预测 (ARIMA模型 - BIC+CV定阶)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(1990, peak_year + 10)

plt.tight_layout()
plt.savefig('arima_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n预测图已保存: arima_forecast.png")

## 5. 总结

In [ ]:
print("="*60)
print("太阳黑子数时间序列分析总结")
print("="*60)
print(f"1. 频谱分析确认太阳活动周主导周期: {dominant_period:.2f} 年")
print(f"   (理论值约为11年)")
print(f"\n2. ARIMA模型 (改进版 - 防止过拟合):")
print(f"   - 差分阶数 d = {d}")
print(f"   - 最优阶数: ARIMA{best_order}")
print(f"   - BIC = {best_bic:.2f}")
print(f"   - 交叉验证 RMSE = {best_cv_rmse:.2f}")
print(f"\n3. 下一太阳活动周峰值预测:")
print(f"   - 峰值年份: {peak_year}")
print(f"   - 预测峰值太阳黑子数: {peak_value:.1f}")
print(f"   - 95%置信区间: [{forecast_ci[peak_idx, 0]:.1f}, {forecast_ci[peak_idx, 1]:.1f}]")
print(f"\n4. 改进措施 (防止过拟合):")
print(f"   ✓ 使用BIC代替AIC（惩罚模型复杂度）")
print(f"   ✓ 时间序列交叉验证（滚动窗口）")
print(f"   ✓ 限制p,q ≤ 4（避免过高阶数）")
print(f"   ✓ Ljung-Box残差检验（验证模型充分性）")
print("="*60)

## 附录：AIC vs BIC 准则对比

| 准则 | 公式 | 特点 | 适用场景 |
|------|------|------|----------|
| **AIC** | $-2\ln(L) + 2k$ | 惩罚较轻，倾向选择复杂模型 | 预测优先 |
| **BIC** | $-2\ln(L) + k\ln(n)$ | 惩罚较重，倾向选择简单模型 | 解释优先，防止过拟合 |

**过拟合的表现：**
- 训练集拟合很好，但测试集预测很差
- 模型阶数过高（p或q很大）
- 残差仍存在自相关

**本项目的改进：**
1. **BIC筛选**：先用BIC选出候选模型（偏向简单）
2. **交叉验证**：用滚动窗口验证模型的真实预测能力
3. **限制阶数**：p和q最大设为4，避免过度复杂
4. **残差检验**：Ljung-Box检验确认模型充分性

## 参考文献

1. SILSO Sunspot Number Data: https://www.sidc.be/SILSO/
2. 太阳活动周: https://en.wikipedia.org/wiki/Solar_cycle
3. ARIMA模型: https://www.statsmodels.org/stable/generated/statsmodels.tsa.arima.model.ARIMA.html
4. AIC/BIC准则: https://en.wikipedia.org/wiki/Akaike_information_criterion
5. 时间序列交叉验证: https://robjhyndman.com/hyndsight/tscv/